# Assignment 5: Energy-Based Models

This notebook covers the foundations of Energy-Based Models from Lecture 5, including:
- Why normalising arbitrary functions is non-trivial and how EBMs handle it
- The Gibbs distribution $p_\theta(x) = \exp(-E_\theta(x)) / Z_\theta$ and the intractable partition function $Z_\theta$
- The MLE gradient and its push-pull interpretation
- Metropolis-Hastings MCMC for sampling from an unnormalised density
- End-to-end training of a parametric 1D EBM and its failure modes

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp
from scipy.stats import norm

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 6)

---
## Part 1: Parameterising Probability Densities

For a function $f_\theta(x)$ to be a valid probability density it must satisfy **two** conditions:

1. **Non-negativity**: $f_\theta(x) \ge 0$ for all $x$
2. **Normalisation**: $\int f_\theta(x)\, dx = 1$

The first condition is easy to enforce — e.g. use $f_\theta(x) = \exp(g_\theta(x))$ for an arbitrary $g_\theta$.  
The second condition is the hard part: for most $g_\theta$ the integral $\int \exp(g_\theta(x))\, dx$ is not 1.

The standard fix is to **divide by the normalisation constant** (partition function):
$$p_\theta(x) = \frac{\exp(g_\theta(x))}{Z_\theta}, \qquad Z_\theta = \int \exp(g_\theta(x'))\, dx'$$

In the EBM convention we write $g_\theta(x) = -E_\theta(x)$, so low energy ↔ high probability.

### 1.1 Numerical Normalisation on a Grid

In [ ]:
def normalise_on_grid(f_vals, dx):
    """Numerically normalise a function evaluated on a grid.

    Args:
        f_vals : np.ndarray (N,), non-negative function values
        dx     : float, grid spacing

    Returns:
        np.ndarray (N,), normalised so integral ≈ 1
    """
    # TODO:
    # 1. Compute Z = sum(f_vals) * dx   (rectangle-rule approximation of the integral)
    # 2. Return f_vals / Z
    Z = np.sum(f_vals) * dx
    return f_vals / Z

In [ ]:
# Sanity check: normalise an unnormalised Gaussian bump
x_grid = np.linspace(-5, 5, 1000)
dx = x_grid[1] - x_grid[0]

# Unnormalised Gaussian (missing the 1/sqrt(2*pi*sigma^2) constant)
sigma = 1.5
f_unnorm = np.exp(-x_grid**2 / (2 * sigma**2))

f_norm = normalise_on_grid(f_unnorm, dx)

integral = np.sum(f_norm) * dx
print(f'Integral of normalised function: {integral:.6f}  (expected 1.0)')

plt.plot(x_grid, f_unnorm / f_unnorm.max(), label='Unnormalised (scaled)', linestyle='--')
plt.plot(x_grid, f_norm, label='Normalised')
plt.plot(x_grid, norm.pdf(x_grid, 0, sigma), label=f'True N(0,{sigma}²)', linestyle=':')
plt.xlabel('x')
plt.ylabel('density')
plt.title('Numerical normalisation on a grid')
plt.legend()
plt.show()

---
## Part 2: Energy-Based Models

An **Energy-Based Model** (EBM) defines a probability density via an energy function $E_\theta : \mathcal{X} \to \mathbb{R}$:

$$p_\theta(x) = \frac{\exp(-E_\theta(x))}{Z_\theta}, \qquad Z_\theta = \int \exp(-E_\theta(x'))\, dx'$$

Key intuitions:
- **Low energy → high probability**: regions where $E_\theta(x)$ is small get assigned large $p_\theta(x)$.
- **High energy → low probability**: regions where $E_\theta(x)$ is large are suppressed.
- The energy can be **any** real-valued function (e.g. a neural network) — non-negativity is guaranteed by the $\exp$, and normalisation is handled by $Z_\theta$.

### 2.1 EBM Density Functions

In [ ]:
def ebm_unnorm_log_density(x, energy_fn):
    """Compute log of unnormalised EBM density: -E(x).

    Args:
        x         : float or np.ndarray
        energy_fn : callable, E(x) -> float or array

    Returns:
        float or np.ndarray, -E(x)
    """
    # TODO: return -energy_fn(x)
    return -energy_fn(x)


def ebm_log_prob(x, energy_fn, log_Z):
    """Compute normalised log probability under the EBM.

    Args:
        x         : float or np.ndarray
        energy_fn : callable, E(x) -> float or array
        log_Z     : float, log partition function

    Returns:
        float or np.ndarray, log p_θ(x)
    """
    # TODO: return ebm_unnorm_log_density(x, energy_fn) - log_Z
    return ebm_unnorm_log_density(x, energy_fn) - log_Z

In [ ]:
# Demo: quadratic energy E_θ(x) = x² / (2σ²) → should recover N(0, σ²)
sigma = 1.5
quadratic_energy = lambda x: x**2 / (2 * sigma**2)

# Analytical log Z for a Gaussian: Z = sqrt(2π σ²)
log_Z_analytical = 0.5 * np.log(2 * np.pi * sigma**2)

# Numerical log Z via grid integration
x_grid = np.linspace(-8, 8, 2000)
dx = x_grid[1] - x_grid[0]
log_Z_numerical = logsumexp(-quadratic_energy(x_grid)) + np.log(dx)

print(f'log Z  analytical: {log_Z_analytical:.6f}')
print(f'log Z  numerical : {log_Z_numerical:.6f}')
print(f'Difference       : {abs(log_Z_analytical - log_Z_numerical):.2e}')

log_p = ebm_log_prob(x_grid, quadratic_energy, log_Z_analytical)
true_log_p = norm.logpdf(x_grid, 0, sigma)

plt.plot(x_grid, np.exp(log_p), label='EBM p_θ(x)')
plt.plot(x_grid, np.exp(true_log_p), linestyle='--', label=f'True N(0,{sigma}²)')
plt.xlabel('x')
plt.ylabel('density')
plt.title('EBM with quadratic energy = Gaussian')
plt.legend()
plt.show()

---
## Part 3: The Partition Function Problem

For a complex, high-dimensional energy function $E_\theta$ the partition function
$$Z_\theta = \int \exp(-E_\theta(x))\, dx$$
is **intractable** — we cannot compute it in closed form.

This creates a fundamental difficulty for training:
- Every time we update $\theta$, $Z_\theta$ changes.
- We cannot evaluate $\log p_\theta(x) = -E_\theta(x) - \log Z_\theta$ exactly.
- Grid-based numerical integration works in 1D but is exponentially expensive in $D$ dimensions.

### 3.1 Estimating log Z Numerically

In [ ]:
def estimate_log_partition_function(energy_fn, x_grid):
    """Numerically estimate log Z via grid integration.

    Args:
        energy_fn : callable, E(x) -> np.ndarray
        x_grid    : np.ndarray (N,), evaluation grid

    Returns:
        float, log Z estimate
    """
    # TODO:
    # 1. Compute log_unnorm = -energy_fn(x_grid)
    # 2. Use logsumexp + log(dx) for numerical stability
    # Hint: from scipy.special import logsumexp
    #       return logsumexp(log_unnorm) + np.log(x_grid[1] - x_grid[0])
    log_unnorm = -energy_fn(x_grid)
    dx = x_grid[1] - x_grid[0]
    return logsumexp(log_unnorm) + np.log(dx)

In [ ]:
# Sanity check: compare numerical vs analytical log Z for Gaussian energy
sigma = 2.0
quadratic_energy = lambda x: x**2 / (2 * sigma**2)
x_grid = np.linspace(-10, 10, 5000)

log_Z_num = estimate_log_partition_function(quadratic_energy, x_grid)
log_Z_ana = 0.5 * np.log(2 * np.pi * sigma**2)
print(f'Numerical log Z : {log_Z_num:.5f}')
print(f'Analytical log Z: {log_Z_ana:.5f}')
print(f'Absolute error  : {abs(log_Z_num - log_Z_ana):.2e}')

### 3.2 Z Changes with θ — The Core Difficulty

In [ ]:
# Bimodal energy: E(x) = -log(exp(-(x-mu)^2/(2s^2)) + exp(-(x+mu)^2/(2s^2)))
# (negative log of a mixture of two Gaussians)
def bimodal_energy(x, mu=2.0, s=0.8):
    log_p = np.logaddexp(
        -0.5 * ((x - mu) / s)**2,
        -0.5 * ((x + mu) / s)**2
    )
    return -log_p

x_grid = np.linspace(-6, 6, 2000)
dx = x_grid[1] - x_grid[0]

# Show Z changes when mu changes
mus = [1.0, 2.0, 3.0, 4.0]
print('Effect of mu on log Z (bimodal energy):')
for mu in mus:
    lZ = estimate_log_partition_function(lambda x: bimodal_energy(x, mu=mu), x_grid)
    print(f'  mu={mu:.1f}  ->  log Z = {lZ:.4f}')

# Plot unnormalised vs normalised density for default mu=2
energy_vals = bimodal_energy(x_grid)
unnorm = np.exp(-energy_vals)
log_Z = estimate_log_partition_function(bimodal_energy, x_grid)
norm_density = np.exp(-energy_vals - log_Z)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(x_grid, unnorm)
axes[0].set_title('Unnormalised density  exp(-E(x))')
axes[0].set_xlabel('x')
axes[1].plot(x_grid, norm_density, color='orange')
axes[1].set_title('Normalised density  p_θ(x)')
axes[1].set_xlabel('x')
plt.suptitle('Bimodal EBM (μ=2, s=0.8)')
plt.tight_layout()
plt.show()

---
## Part 4: MLE Gradient and Push-Pull Dynamics

The MLE objective is $\max_\theta \frac{1}{N}\sum_i \log p_\theta(x_i)$.  Taking the gradient:

$$\nabla_\theta \log p_\theta(x) = -\nabla_\theta E_\theta(x) + \mathbb{E}_{x' \sim p_\theta}[\nabla_\theta E_\theta(x')]$$

This has a beautiful **push-pull** interpretation:
- **Negative phase (push data down)**: $-\nabla_\theta E_\theta(x_{\text{data}})$ — reduce the energy at observed data points.
- **Positive phase (pull model up)**: $+\mathbb{E}_{x' \sim p_\theta}[\nabla_\theta E_\theta(x')]$ — increase the energy at points sampled from the current model.

For a **linear-in-parameters** energy $E_\theta(x) = \theta \cdot \phi(x)$ the gradient simplifies to:

$$\nabla_\theta \log p_\theta(x) = -\phi(x_{\text{data}}) + \mathbb{E}_{x' \sim p_\theta}[\phi(x')]$$

### 4.1 MLE Gradient Estimator

In [ ]:
def ebm_mle_gradient(x_data, x_model_samples, phi):
    """Compute the MLE gradient for a linear-in-parameters EBM.

    E_θ(x) = θ · φ(x), so ∇_θ log p_θ(x) = -φ(x_data) + mean(φ(x_model))

    Args:
        x_data          : np.ndarray (N,), observed data
        x_model_samples : np.ndarray (M,), samples from current model p_θ
        phi             : callable, feature function φ(x) -> float or array

    Returns:
        float, gradient estimate (scalar for scalar phi)
    """
    # TODO:
    # 1. Compute data term:  -mean(phi(x_data))
    # 2. Compute model term:  mean(phi(x_model_samples))
    # 3. Return data_term + model_term
    data_term  = -np.mean(phi(x_data))
    model_term =  np.mean(phi(x_model_samples))
    return data_term + model_term

In [ ]:
# Sanity check: with E_θ(x) = θ x², the feature is φ(x) = x²
# If data ~ N(0,1) and model ~ N(0,1), gradient should be ≈ 0 (convergence)
np.random.seed(0)
x_data   = np.random.randn(500)       # data from N(0,1)
x_model  = np.random.randn(500)       # model also N(0,1) -> should be at optimum
phi_quad = lambda x: x**2

grad = ebm_mle_gradient(x_data, x_model, phi_quad)
print(f'Gradient (at convergence, should be ≈ 0): {grad:.4f}')

# Now illustrate push-pull: data ~ N(0,1), model ~ N(0,2)
x_model_wide = np.random.randn(500) * 2   # model too spread out
grad_wide = ebm_mle_gradient(x_data, x_model_wide, phi_quad)
print(f'Gradient (model too wide, should be > 0): {grad_wide:.4f}')
print('(Positive gradient -> increase θ -> steeper energy -> model narrows)')

### 4.2 Visualising Push-Pull Dynamics

In [ ]:
# With E_θ(x) = θ x², show how the energy and density change with θ
x_plot = np.linspace(-4, 4, 400)
thetas = [0.1, 0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for theta in thetas:
    E_fn   = lambda x, t=theta: t * x**2
    log_Z  = estimate_log_partition_function(E_fn, x_plot)
    energy = E_fn(x_plot)
    prob   = np.exp(-energy - log_Z)
    axes[0].plot(x_plot, energy, label=f'θ={theta}')
    axes[1].plot(x_plot, prob,   label=f'θ={theta}')

axes[0].set_title('Energy E_θ(x) = θ x²')
axes[0].set_xlabel('x')
axes[0].set_ylabel('E(x)')
axes[0].legend()

axes[1].set_title('Normalised density p_θ(x)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('p(x)')
axes[1].legend()

plt.suptitle('Push-pull: larger θ → steeper energy → narrower density')
plt.tight_layout()
plt.show()

---
## Part 5: Metropolis-Hastings Sampling

To evaluate the positive-phase gradient $\mathbb{E}_{x' \sim p_\theta}[\nabla_\theta E_\theta(x')]$ we need samples from $p_\theta$.  
But $p_\theta(x) \propto \exp(-E_\theta(x))$ is only known up to $Z_\theta$.

**Metropolis-Hastings (MH)** solves this: the acceptance ratio only requires the ratio $p_\theta(x')/p_\theta(x)$, and $Z_\theta$ **cancels**:

$$\alpha = \min\!\left(1,\, \frac{p_\theta(x')}{p_\theta(x)}\right) = \min\!\left(1,\, \exp\bigl(E_\theta(x) - E_\theta(x')\bigr)\right)$$

Algorithm (with Gaussian proposals $x' = x + \mathcal{N}(0, \sigma_{\text{prop}}^2)$):
1. Initialise $x_0$.
2. Propose $x' = x + \mathcal{N}(0, \sigma_{\text{prop}}^2)$.
3. Compute $\log \alpha = E_\theta(x) - E_\theta(x')$.
4. Draw $u \sim \text{Uniform}(0,1)$; if $\log u < \log \alpha$ set $x \leftarrow x'$.
5. Record $x$; repeat.

### 5.1 Implementing Metropolis-Hastings

In [ ]:
def metropolis_hastings(energy_fn, x_init, proposal_std, n_steps):
    """Run Metropolis-Hastings Markov chain.

    Proposal: x' = x + N(0, proposal_std²)
    Acceptance: α = min(1, exp(-(E(x') - E(x))))

    Args:
        energy_fn    : callable, E(x) -> float
        x_init       : float, starting point
        proposal_std : float, standard deviation of Gaussian proposal
        n_steps      : int, number of MCMC steps

    Returns:
        chain : np.ndarray (n_steps,), all states (accepted or carried forward)
    """
    # TODO:
    # 1. Initialise x = x_init, chain = []
    # 2. For each step:
    #    a. Propose x' = x + np.random.randn() * proposal_std
    #    b. Compute log_alpha = -(energy_fn(x') - energy_fn(x))
    #    c. Accept x' with probability min(1, exp(log_alpha)):
    #       u ~ Uniform(0,1); if log(u) < log_alpha: x = x'
    #    d. Append x to chain
    # 3. Return np.array(chain)
    x = x_init
    chain = []
    for _ in range(n_steps):
        x_prop    = x + np.random.randn() * proposal_std
        log_alpha = -(energy_fn(x_prop) - energy_fn(x))
        if np.log(np.random.rand()) < log_alpha:
            x = x_prop
        chain.append(x)
    return np.array(chain)

In [ ]:
# Sanity check: quadratic energy → should recover N(0,1)
np.random.seed(42)
energy_standard_normal = lambda x: 0.5 * x**2

chain = metropolis_hastings(energy_standard_normal, x_init=0.0, proposal_std=1.0, n_steps=10000)

print(f'Chain mean: {chain.mean():.3f}  (expected ≈ 0.0)')
print(f'Chain std : {chain.std():.3f}  (expected ≈ 1.0)')

x_plot = np.linspace(-4, 4, 300)
plt.hist(chain, bins=60, density=True, alpha=0.6, label='MH samples')
plt.plot(x_plot, norm.pdf(x_plot, 0, 1), 'r-', lw=2, label='True N(0,1)')
plt.xlabel('x')
plt.title('MH sampling from N(0,1) via quadratic energy')
plt.legend()
plt.show()

### 5.2 Bimodal Energy

In [ ]:
# Bimodal: mixture of N(-2, 0.5²) and N(2, 0.5²)
def bimodal_energy_mh(x, mu=2.0, s=0.5):
    """Energy for a symmetric bimodal distribution."""
    return -np.logaddexp(
        -0.5 * ((x - mu) / s)**2,
        -0.5 * ((x + mu) / s)**2
    )

np.random.seed(42)
chain_bimodal = metropolis_hastings(
    bimodal_energy_mh, x_init=0.0, proposal_std=1.5, n_steps=20000
)

# True density (unnormalised -> normalised on grid)
x_plot = np.linspace(-5, 5, 1000)
dx = x_plot[1] - x_plot[0]
true_density = normalise_on_grid(np.exp(-bimodal_energy_mh(x_plot)), dx)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Chain trajectory
axes[0].plot(chain_bimodal[:500], alpha=0.7, lw=0.8)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('x')
axes[0].set_title('MH chain trajectory (first 500 steps)')

# Histogram vs true density
axes[1].hist(chain_bimodal, bins=80, density=True, alpha=0.6, label='MH samples')
axes[1].plot(x_plot, true_density, 'r-', lw=2, label='True density')
axes[1].set_xlabel('x')
axes[1].set_title('MH histogram vs true bimodal density')
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.3 Effect of Chain Length

In [ ]:
# Compare chain lengths: 100 vs 1000 vs 10000 steps
np.random.seed(42)
n_steps_list = [100, 1000, 10000]

x_plot = np.linspace(-5, 5, 1000)
dx = x_plot[1] - x_plot[0]
true_density = normalise_on_grid(np.exp(-bimodal_energy_mh(x_plot)), dx)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, n in zip(axes, n_steps_list):
    chain = metropolis_hastings(bimodal_energy_mh, x_init=0.0, proposal_std=1.5, n_steps=n)
    ax.hist(chain, bins=40, density=True, alpha=0.6, label=f'MH ({n} steps)')
    ax.plot(x_plot, true_density, 'r-', lw=2, label='True density')
    ax.set_title(f'{n} steps')
    ax.set_xlabel('x')
    ax.legend(fontsize=8)

plt.suptitle('Effect of chain length on MH sample quality')
plt.tight_layout()
plt.show()

---
## Part 6: Training a 1D EBM

We now train an EBM with a **parametric energy**:
$$E_\theta(x) = \theta_1 x^2 + \theta_2 x^4, \qquad \theta = (\theta_1, \theta_2)$$

The true data distribution is a mixture of two Gaussians: $\frac{1}{2}\mathcal{N}(-2, 0.5^2) + \frac{1}{2}\mathcal{N}(2, 0.5^2)$.

At each gradient step:
1. Run an MH chain to get samples $x' \sim p_\theta$ (positive phase).
2. Compute gradient $g_k = -\text{mean}(\phi_k(X_{\text{data}})) + \text{mean}(\phi_k(x'))$ for $\phi_0(x)=x^2$, $\phi_1(x)=x^4$.
3. Update $\theta \leftarrow \theta + \text{lr} \cdot g$.

In [ ]:
# Generate training data: mixture of two Gaussians
np.random.seed(42)
n_data = 500
components = np.random.choice([-1, 1], size=n_data)  # which mode
X_data = components * 2.0 + np.random.randn(n_data) * 0.5

x_plot = np.linspace(-5, 5, 500)
true_density = 0.5 * norm.pdf(x_plot, -2, 0.5) + 0.5 * norm.pdf(x_plot, 2, 0.5)

plt.hist(X_data, bins=50, density=True, alpha=0.5, label='Training data')
plt.plot(x_plot, true_density, 'r-', lw=2, label='True mixture density')
plt.xlabel('x')
plt.title('Training data: mixture of two Gaussians')
plt.legend()
plt.show()

In [ ]:
def train_ebm(X_data, n_iter=200, lr=0.01, n_mcmc_steps=500, proposal_std=0.5):
    """Train a 1D EBM with energy E_θ(x) = θ[0]*x² + θ[1]*x⁴.

    Args:
        X_data        : np.ndarray (N,), training data
        n_iter        : int, gradient ascent steps
        lr            : float, learning rate
        n_mcmc_steps  : int, MH chain length per gradient step
        proposal_std  : float, MH proposal standard deviation

    Returns:
        theta_history : np.ndarray (n_iter, 2), parameter trajectory
        ll_history    : list of float, approximate average log-likelihoods
    """
    theta = np.array([1.0, 0.1])   # initial parameters
    theta_history = []
    ll_history = []

    x_grid = np.linspace(-6, 6, 1000)

    for i in range(n_iter):
        # TODO:
        # 1. Define energy function for current theta
        # 2. Run MH chain from a random data point to get x_model ~ p_θ
        # 3. Compute gradient for each parameter:
        #    phi_0(x) = x²,  phi_1(x) = x⁴
        #    g[k] = ebm_mle_gradient(X_data, x_model, phi_k)
        # 4. Gradient ascent: theta += lr * g
        # 5. Estimate log Z numerically; compute average log-likelihood
        # 6. Append theta.copy() and ll to histories
        raise NotImplementedError

    return np.array(theta_history), ll_history

In [ ]:
# Run training — this cell will raise NotImplementedError until train_ebm is implemented
np.random.seed(42)
theta_history, ll_history = train_ebm(
    X_data, n_iter=200, lr=0.005, n_mcmc_steps=500, proposal_std=0.8
)

print(f'Initial theta: [1.0, 0.1]')
print(f'Final   theta: {theta_history[-1]}')

In [ ]:
# Plot learned density at iterations 1, 50, 200 vs true density
x_plot = np.linspace(-6, 6, 500)
dx = x_plot[1] - x_plot[0]
true_density = 0.5 * norm.pdf(x_plot, -2, 0.5) + 0.5 * norm.pdf(x_plot, 2, 0.5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, idx in zip(axes, [0, 49, 199]):
    t = theta_history[idx]
    E_fn = lambda x, t=t: t[0] * x**2 + t[1] * x**4
    log_Z = estimate_log_partition_function(E_fn, x_plot)
    p = np.exp(-E_fn(x_plot) - log_Z)
    ax.plot(x_plot, p, lw=2, label=f'EBM iter {idx+1}')
    ax.plot(x_plot, true_density, 'r--', lw=1.5, label='True density')
    ax.set_title(f'Iteration {idx+1}  θ=[{t[0]:.2f}, {t[1]:.3f}]')
    ax.set_xlabel('x')
    ax.legend(fontsize=8)

plt.suptitle('EBM learned density over training')
plt.tight_layout()
plt.show()

# Log-likelihood curve
plt.figure()
plt.plot(ll_history)
plt.xlabel('Iteration')
plt.ylabel('Avg log-likelihood')
plt.title('Training log-likelihood')
plt.show()

---
## Part 7: Failure Modes

MCMC-based EBM training has two well-known failure modes:

1. **Short chains → biased negative phase gradient**  
   If the MH chain is too short (few steps), the samples $x' \sim p_\theta$ are highly correlated with the initialisation point and do not represent the true model distribution. The positive-phase gradient is biased, leading to slow or incorrect convergence.

2. **Multimodal targets → mode collapse**  
   With a small proposal standard deviation, the MH chain mixes slowly across modes. If the chain starts near one mode, it may never visit the other modes within the allotted steps. The positive-phase gradient only "sees" one mode, causing the model to focus on that region.

In [ ]:
# TODO (exploration): Demonstrate both failure modes:
#
# 1. SHORT CHAIN FAILURE
#    Train with n_mcmc_steps=10 and compare to n_mcmc_steps=1000.
#    Plot the log-likelihood curves and final learned densities side by side.
#    Show that short chains lead to slower or poorer convergence.
#
# 2. MULTIMODAL MODE COLLAPSE
#    Generate data from a trimodal distribution:
#      0.33 * N(-4, 0.5²) + 0.33 * N(0, 0.5²) + 0.33 * N(4, 0.5²)
#    Train with proposal_std=0.3 (small) and proposal_std=2.0 (large).
#    Show that the small proposal_std chain gets stuck in one mode.
#
# Tip: reuse train_ebm once you have implemented it.

raise NotImplementedError

**Question:** What is the *negative phase gradient* in EBM training, and why is it computationally expensive to estimate accurately?

**Your answer**: TODO

**Question:** What solutions exist for the two failure modes described above? Name at least one remedy for each.

**Your answer**: TODO

---
## Summary

In this notebook you:
- Saw why normalising arbitrary functions is hard and why EBMs use the Gibbs form $p_\theta(x) = \exp(-E_\theta(x)) / Z_\theta$
- Implemented energy-to-probability conversion with numerical partition function estimation via `logsumexp`
- Derived the push-pull MLE gradient for EBMs and verified its behaviour
- Implemented Metropolis-Hastings MCMC sampling and demonstrated it on unimodal and bimodal targets
- Trained a parametric 1D EBM ($E_\theta(x) = \theta_1 x^2 + \theta_2 x^4$) end-to-end on mixture data
- Identified two key failure modes of MCMC-based training: short-chain bias and multimodal mixing

**Next**: Assignment 6 covers Diffusion Models, which sidestep the MCMC problem by framing generation as iterative denoising.